# Prompt Testing Notebook
Test and iterate on prompts for email classification and processing

In [ ]:
import sys
sys.path.append('..')

from src.llm.openai_client import OpenAIClient
from prompt_engineering.templates import PromptTemplate, PromptManager
from config.config_loader import get_model_config

## Setup LLM Client

In [ ]:
config = get_model_config()
azure_config = config.get("azure_openai", {})

client = OpenAIClient(
    api_key=azure_config.get("api_key"),
    model=azure_config.get("chat_model", {}).get("deployment_name"),
    use_azure=True,
    azure_endpoint=azure_config.get("endpoint")
)

## Test Email Classification Prompts

In [ ]:
# Test email samples
test_emails = [
    {
        "subject": "Happy Birthday!",
        "body": "Wishing you a wonderful birthday filled with joy and happiness!"
    },
    {
        "subject": "Approval Required: Budget Request",
        "body": "Please review and approve the attached budget request for Q4."
    },
    {
        "subject": "50% OFF - Limited Time Offer!",
        "body": "Don't miss out on our biggest sale of the year. Shop now!"
    }
]

In [ ]:
# Classification prompt template
classification_template = PromptTemplate(
    template="""
    Classify the following email into one of these categories:
    - Birthday: Birthday or anniversary wishes
    - Actionable: Requires action (approval, meeting, etc.)
    - Promotional: Marketing or promotional content
    
    Email Subject: ${subject}
    Email Body: ${body}
    
    Respond with only the category name and confidence (0-1).
    Format: Category: <name>, Confidence: <score>
    """,
    variables=["subject", "body"]
)

# Test classification
for email in test_emails:
    prompt = classification_template.format(**email)
    response = client.generate(prompt, temperature=0.3, max_tokens=50)
    print(f"Subject: {email['subject']}")
    print(f"Classification: {response}")
    print("-" * 50)

## Test Birthday Email Drafting

In [ ]:
birthday_template = PromptTemplate(
    template="""
    Draft a ${tone} birthday email for ${recipient_name}.
    ${context}
    
    Include:
    - Warm greeting
    - Birthday wishes
    - Personal touch
    - Professional closing
    """,
    variables=["recipient_name", "tone", "context"]
)

# Test different tones
test_cases = [
    {"recipient_name": "John", "tone": "professional", "context": "He is a colleague in the marketing team."},
    {"recipient_name": "Sarah", "tone": "warm and friendly", "context": "She is a close team member."},
]

for case in test_cases:
    prompt = birthday_template.format(**case)
    response = client.generate(prompt, temperature=0.8, max_tokens=200)
    print(f"Recipient: {case['recipient_name']} ({case['tone']})")
    print(response)
    print("=" * 50)

## Test Actionable Email Response

In [ ]:
actionable_template = PromptTemplate(
    template="""
    Analyze this actionable email and suggest a response:
    
    Subject: ${subject}
    Body: ${body}
    
    Provide:
    1. Type of action required
    2. Suggested response
    3. Key points to address
    """,
    variables=["subject", "body"]
)

actionable_email = {
    "subject": "Meeting Request - Project Review",
    "body": "Can we schedule a meeting next week to review the project progress? I'm available Tuesday and Thursday afternoons."
}

prompt = actionable_template.format(**actionable_email)
response = client.generate(prompt, temperature=0.7, max_tokens=300)
print(response)